In [1]:
import os

In [2]:
%pwd

'd:\\PythonProjects\\E2E-ML-Project-with-mlflow\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\PythonProjects\\E2E-ML-Project-with-mlflow'

In [5]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/mbdev1993/E2E-ML-Project-with-mlflow.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="mbdev1993"
os.environ["MLFLOW_TRACKING_PASSWORD"]="8244e1267a62c7dfa8cac5777c4b627e5fb48ea0"

In [46]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path : Path
    model_data : Path
    all_params : dict
    metric_file_name: Path
    target_column : str
    mlflow_uri : str


In [47]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [48]:
class configurationManager:
    def __init__(self, 
                 config_filepath=CONFIG_FILE_PATH, 
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN
        model_eval_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_data=config.model_path,
            all_params=params,
            metric_file_name=config.metric_file_name,
            target_column=schema.name,
            mlflow_uri="https://dagshub.com/mbdev1993/E2E-ML-Project-with-mlflow.mlflow"
        )
        return model_eval_config

In [49]:
import os
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [50]:
from utils.common import save_json


class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, predicted):
        r2 = r2_score(actual, predicted)
        rmse = np.sqrt(mean_squared_error(actual, predicted))
        mae = mean_absolute_error(actual, predicted)
        return r2, rmse, mae
    
    def log_into_mlflow(self):
        Path(self.config.root_dir).mkdir(parents=True, exist_ok=True)

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_data)

        test_x = test_data.drop(self.config.target_column, axis=1) 
        test_y = test_data[self.config.target_column]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            predicted_qualities = model.predict(test_x)
            (r2, rmse, mae) = self.eval_metrics(test_y, predicted_qualities)

            scores = {"r2_score": r2,"rmse": rmse,
                "mae": mae
            }
            
            Path(self.config.root_dir).mkdir(parents=True, exist_ok=True)

            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)
            mlflow.log_metric("r2_score", r2)
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("mae", mae)

            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetWineModel")
            else:
                mlflow.sklearn.log_model(model, "model")

In [51]:
try:
    config = configurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-07-01 07:56:17,461: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-01 07:56:17,463: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-01 07:56:17,467: INFO: common: yaml file: config\schema.yaml loaded successfully]
[2026-07-01 07:56:17,469: INFO: common: created directory at: artifacts]


Registered model 'ElasticnetWineModel' already exists. Creating a new version of this model...
2026/07/01 07:56:26 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: ElasticnetWineModel, version 2
Created version '2' of model 'ElasticnetWineModel'.
